# 04 — Hierarchical cell typing, state, and QuPath export

This notebook inspects production broad and specific cell assignments, orthogonal state annotations, and the uncertainty-preserving QuPath export. All assignment logic lives in the versioned typing registry and production APIs; the notebook does not recreate gates or classifiers.

In [ ]:
import json

import pandas as pd
from IPython.display import display

from phenocycler.artifacts import StageManifest
from phenocycler.config import load_config
from phenocycler.expression import read_single_partition
from phenocycler.pipeline import (
    RunContext,
    export_qupath,
    resolve_run_context,
    run_stage,
    status,
)

CONFIG_PATH = None
cfg = load_config(CONFIG_PATH)
context: RunContext = resolve_run_context(cfg)
print(f"run_id={context.run_id}  donors={len(context.donors)}  root={context.run_root}")

In [ ]:
status_code = status(context)
print(f"status return code: {status_code}")

## Assignment contract

- **Broad identity** is evaluated simultaneously across the registered broad classes. Ordered notebook gates do not decide the winner.
- **Specific identity** is evaluated only inside an accepted parent broad class. A parent without a supported subtype remains explicitly unclassified at that level.
- **Ranked alternatives** retain the competing broad-class probabilities as JSON instead of discarding all but the label.
- **Assignment status and reasons** distinguish authoritative anchors, supported inference, ambiguity, unavailable evidence, and `Other`. `Other` requires valid negative evidence; missing evidence does not become negative.
- **State** (currently Ki67/PCNA proliferation) is orthogonal to identity and never changes a broad or specific type.
- **QuPath export** preserves object identity, image, broad and specific labels, status, confidence, best broad alternative, and ranked alternatives. Compatibility aliases are included for older QuPath consumers.

## Optional production execution

`type` consumes calibrated evidence. `states` consumes selected expression independently. Existing valid stages are checked rather than recomputed.

In [ ]:
RUN_STAGES = False

if RUN_STAGES:
    for stage_name in ("type", "states"):
        run_stage(context, stage_name)
else:
    print("Inspection only. Set RUN_STAGES=True to run production typing and state annotation.")

In [ ]:
manifest_rows = []
for stage_name in ("type", "states"):
    path = context.stage_manifest_path(stage_name)
    if path.exists():
        manifest = StageManifest.read_json(path)
        manifest_rows.append({
            "stage": stage_name,
            "method_version": manifest.method_version,
            "donors": len(manifest.completed_donors),
            "rows": manifest.output.total_rows,
            "columns": len(manifest.output.schema),
            "schema": manifest.output.schema_sha256[:12],
            "objects": manifest.output.object_id_sha256[:12],
            "content": manifest.content_id[:12],
        })
display(pd.DataFrame(manifest_rows))

## Inspect hierarchical assignments

Review assignment statuses and reasons alongside labels. A label count alone cannot distinguish an anchored call from an ambiguous or evidence-limited result.

In [ ]:
DONOR = context.donors[0]
type_manifest = context.stage_manifest_path("type")

if type_manifest.exists():
    assignments = read_single_partition(context.config.assignments_dir, DONOR)
    print(f"donor {DONOR}: {len(assignments):,} assignments")
    display(
        assignments.groupby(["broad_type", "broad_assignment_status"], dropna=False)
        .size().rename("cells").sort_values(ascending=False).to_frame()
    )
    display(
        assignments.groupby(["specific_type", "assignment_status"], dropna=False)
        .size().rename("cells").sort_values(ascending=False).to_frame()
    )
    review_columns = [
        column for column in (
            "object_id", "broad_type", "specific_type", "assignment_status",
            "confidence", "best_broad_type", "broad_reason", "subtype_reason",
            "ranked_probabilities", "ranked_subtype_probabilities"
        ) if column in assignments
    ]
    display(assignments.loc[:, review_columns].head(30))
else:
    assignments = pd.DataFrame()
    print("Typing artifacts are not complete yet.")

In [ ]:
if not assignments.empty and "ranked_probabilities" in assignments:
    ranked_rows = assignments.loc[
        assignments["ranked_probabilities"].astype(str).ne("[]"),
        ["object_id", "broad_type", "assignment_status", "ranked_probabilities"],
    ].head(10).copy()
    ranked_rows["ranked_alternatives"] = ranked_rows["ranked_probabilities"].map(json.loads)
    display(ranked_rows.drop(columns="ranked_probabilities"))
else:
    print("No ranked alternatives are available to inspect.")

## Inspect orthogonal state

State tables share donor/object keys with the identity tables but remain separate artifacts. Their thresholds and model status are repeated for explicit provenance.

In [ ]:
states_manifest = context.stage_manifest_path("states")
if states_manifest.exists():
    states = read_single_partition(context.config.state_dir, DONOR)
    state_columns = [column for column in states if column.endswith("__state")]
    display(states.head(20))
    state_counts = {
        column.removesuffix("__state"): states[column].value_counts(dropna=False).to_dict()
        for column in state_columns
    }
    display(pd.DataFrame(state_counts).fillna(0).astype(int))
    state_audit_path = context.config.audit_dir / "state_models.parquet"
    if state_audit_path.exists():
        display(pd.read_parquet(state_audit_path).loc[lambda frame: frame["donor_id"].astype(str).eq(DONOR)])
else:
    print("State artifacts are not complete yet.")

## Optional QuPath export

Export is a production API call and is disabled by default. When a current export already exists, the API validates its fingerprints instead of rewriting it.

In [ ]:
EXPORT_QUPATH = False

if EXPORT_QUPATH:
    export_qupath(context)
else:
    export_manifest_path = context.config.manifests_dir / "qupath_export.json"
    exported_files = sorted(context.config.qupath_class_dir.glob("*.csv"))
    print(f"export manifest: {export_manifest_path if export_manifest_path.exists() else 'not present'}")
    print(f"exported donor CSVs: {len(exported_files)}")

## Review checklist

Before using labels downstream, confirm that every donor is present, object counts match upstream manifests, assignment statuses and reasons are plausible, ranked alternatives are retained, state is not conflated with identity, and the QuPath export manifest belongs to the current content-addressed run.